# 00 — Does your environment work?

**2026 Quantum Accelerator · Day 1 · 09:45 · 10 minutes**

Two cells. If both print what they say they will, you are ready for the day. If either
does not, put your hand up now rather than at 13:00.

### Step 1 — copy today's notebooks into your own folder

The `shared` folder is shared with everyone in the room. Work in your own copy, or
twenty-five people will be editing the same file.

In [ ]:
import shutil
from pathlib import Path

SOURCE = Path("../shared/day1")        # adjust if your instance mounts it elsewhere
TARGET = Path.home() / "day1"

if SOURCE.exists():
    shutil.copytree(SOURCE, TARGET, dirs_exist_ok=True)
    print(f"copied to {TARGET}")
    for p in sorted(TARGET.iterdir()):
        print("   ", p.name)
else:
    print(f"{SOURCE} not found -- you may already be in your own copy. "
          f"Check with the facilitator.")

### Step 2 — the stack

In [ ]:
import sys
import importlib.metadata as md

print("python", sys.version.split()[0])
for pkg in ("pyqubo", "dimod", "dwave-samplers", "ortools",
            "networkx", "numpy", "matplotlib"):
    try:
        print(f"  {pkg:<16} {md.version(pkg)}")
    except Exception:
        print(f"  {pkg:<16} *** MISSING -- tell the facilitator ***")

### Step 3 — solve something

A two-variable problem: pick exactly one of two options. The expected output is printed
below the cell — compare it with what you get.

In [ ]:
from pyqubo import Array, Constraint, Placeholder
from dwave.samplers import SimulatedAnnealingSampler
import dimod

x = Array.create("x", shape=2, vartype="BINARY")
model = (Placeholder("lam") * Constraint((x[0] + x[1] - 1) ** 2, label="pick_one")).compile()

feed = {"lam": 1.0}
bqm = model.to_bqm(feed_dict=feed)

exact = dimod.ExactSolver().sample(bqm)
annealed = SimulatedAnnealingSampler().sample(bqm, num_reads=20, num_sweeps=100, seed=1)
decoded = model.decode_sampleset(annealed, feed_dict=feed)

legal = [d for d in decoded if not d.constraints(only_broken=True)]
print(f"variables in the model        : {len(bqm.variables)}")
print(f"exact solver, lowest energy   : {exact.first.energy:.1f}")
print(f"annealing runs that were legal: {len(legal)} of {len(decoded)}")
print(f"one legal answer              : x = "
      f"{[legal[0].array('x', i) for i in range(2)]}" if legal else "no legal answer")
print()
print("ENVIRONMENT OK" if legal and abs(exact.first.energy) < 1e-9 else "SOMETHING IS WRONG")

**Expected:**

```
variables in the model        : 2
exact solver, lowest energy   : 0.0
annealing runs that were legal: 20 of 20
one legal answer              : x = [1, 0]     (or [0, 1] -- both are correct)

ENVIRONMENT OK
```

`x = [1, 0]` and `x = [0, 1]` are equally good answers, so either may appear. What matters
is that the last line says `ENVIRONMENT OK`.